In [1]:
import pandas as pd

In [3]:
prod = pd.read_csv("../data/raw/production_consumption_DK2_2024_08_10_2025_10_01.csv")

prod.head()

,HourUTC,HourDK,PriceArea,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,LocalPowerSelfConMWh,OffshoreWindLt100MW_MWh,OffshoreWindGe100MW_MWh,OnshoreWindLt50kW_MWh,...,ExchangeSE_MWh,ExchangeGE_MWh,ExchangeNL_MWh,ExchangeGB_MWh,ExchangeGreatBelt_MWh,GrossConsumptionMWh,GridLossTransmissionMWh,GridLossInterconnectorsMWh,GridLossDistributionMWh,PowerToHeatMWh
0,2024-08-09T22:00:00,2024-08-10T00:00:00,DK2,0.0,32.321104,89.469661,2.364408,20.4316,493.545862,0.765346,...,1255.704266,-672.892992,NaN,NaN,-243.0,1288.725360,37.779306,14.816992,51.277244,68.201319
1,2024-08-09T23:00:00,2024-08-10T01:00:00,DK2,0.0,32.775521,88.280820,1.476173,17.5967,461.329318,0.623995,...,1201.172750,-682.668008,NaN,NaN,-201.0,1220.577169,36.957540,15.395008,50.056950,65.264387
2,2024-08-10T00:00:00,2024-08-10T02:00:00,DK2,0.0,33.634796,85.318927,1.499870,17.6179,442.736322,0.654639,...,1234.965258,-685.675992,NaN,NaN,-237.7,1209.086682,36.256258,15.454992,50.302570,75.440486
3,2024-08-10T01:00:00,2024-08-10T03:00:00,DK2,0.0,34.718886,85.439154,1.587277,13.2659,369.858196,0.638296,...,1249.801250,-711.274992,NaN,NaN,-155.8,1177.828871,33.454844,15.386992,47.926672,77.422757
4,2024-08-10T02:00:00,2024-08-10T04:00:00,DK2,0.0,32.970704,83.689183,1.550827,11.3103,314.532159,0.579736,...,1207.414750,-733.964984,NaN,NaN,-64.0,1136.127061,30.032715,15.405984,46.527364,68.731282


In [4]:
prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   HourUTC                     10000 non-null  object 
 1   HourDK                      10000 non-null  object 
 2   PriceArea                   10000 non-null  object 
 3   CentralPowerMWh             10000 non-null  float64
 4   LocalPowerMWh               10000 non-null  float64
 5   CommercialPowerMWh          10000 non-null  float64
 6   LocalPowerSelfConMWh        10000 non-null  float64
 7   OffshoreWindLt100MW_MWh     10000 non-null  float64
 8   OffshoreWindGe100MW_MWh     10000 non-null  float64
 9   OnshoreWindLt50kW_MWh       10000 non-null  float64
 10  OnshoreWindGe50kW_MWh       10000 non-null  float64
 11  HydroPowerMWh               10000 non-null  float64
 12  SolarPowerLt10kW_MWh        10000 non-null  float64
 13  SolarPowerGe10Lt40kW_MWh    1000

In [5]:
prod["HourDK"] = pd.to_datetime(prod["HourDK"])

print("Rows:", len(prod))
print("Date range:", prod["HourDK"].min(), "→", prod["HourDK"].max())
print("Price areas:", prod["PriceArea"].unique())

Rows: 10000
Date range: 2024-08-10 00:00:00 → 2025-09-30 15:00:00
Price areas: ['DK2']


In [6]:
prod["offshore_wind_mwh"] = (
    prod["OffshoreWindLt100MW_MWh"] +
    prod["OffshoreWindGe100MW_MWh"]
)

prod["onshore_wind_mwh"] = (
    prod["OnshoreWindLt50kW_MWh"] +
    prod["OnshoreWindGe50kW_MWh"]
)

prod["solar_mwh"] = (
    prod["SolarPowerLt10kW_MWh"] +
    prod["SolarPowerGe10Lt40kW_MWh"] +
    prod["SolarPowerGe40kW_MWh"] +
    prod["SolarPowerSelfConMWh"]
)

prod["total_wind_mwh"] = (
    prod["offshore_wind_mwh"] +
    prod["onshore_wind_mwh"]
)

prod["renewable_generation_mwh"] = (
    prod["total_wind_mwh"] +
    prod["solar_mwh"]
)

prod["net_load_mwh"] = (
    prod["GrossConsumptionMWh"] -
    prod["renewable_generation_mwh"]
)

In [7]:
prod_clean = prod[
    [
        "HourDK",
        "HourUTC",
        "PriceArea",
        "GrossConsumptionMWh", #L
        "CentralPowerMWh", #G
        "LocalPowerMWh", #G
        "CommercialPowerMWh", #G
        "offshore_wind_mwh",
        "onshore_wind_mwh",
        "solar_mwh",
        "total_wind_mwh",
        "renewable_generation_mwh",
        "net_load_mwh",
        "ExchangeSE_MWh",
        "ExchangeGE_MWh",
        "ExchangeGreatBelt_MWh",
        "PowerToHeatMWh",
    ]
].copy()

prod_clean.head()

,HourDK,HourUTC,PriceArea,GrossConsumptionMWh,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,offshore_wind_mwh,onshore_wind_mwh,solar_mwh,total_wind_mwh,renewable_generation_mwh,net_load_mwh,ExchangeSE_MWh,ExchangeGE_MWh,ExchangeGreatBelt_MWh,PowerToHeatMWh
0,2024-08-10 00:00:00,2024-08-09T22:00:00,DK2,1288.725360,0.0,32.321104,89.469661,513.977462,308.117284,0.285386,822.094746,822.380132,466.345228,1255.704266,-672.892992,-243.0,68.201319
1,2024-08-10 01:00:00,2024-08-09T23:00:00,DK2,1220.577169,0.0,32.775521,88.280820,478.926018,299.629530,0.229564,778.555548,778.785112,441.792057,1201.172750,-682.668008,-201.0,65.264387
2,2024-08-10 02:00:00,2024-08-10T00:00:00,DK2,1209.086682,0.0,33.634796,85.318927,460.354222,314.615082,0.202579,774.969304,775.171883,433.914799,1234.965258,-685.675992,-237.7,75.440486
3,2024-08-10 03:00:00,2024-08-10T01:00:00,DK2,1177.828871,0.0,34.718886,85.439154,383.124096,288.741180,0.190264,671.865276,672.055540,505.773331,1249.801250,-711.274992,-155.8,77.422757
4,2024-08-10 04:00:00,2024-08-10T02:00:00,DK2,1136.127061,0.0,32.970704,83.689183,325.842459,281.584758,0.185393,607.427217,607.612610,528.514451,1207.414750,-733.964984,-64.0,68.731282


In [8]:
prod_clean.isna().sum()

HourDK                      0
HourUTC                     0
PriceArea                   0
GrossConsumptionMWh         0
CentralPowerMWh             0
LocalPowerMWh               0
CommercialPowerMWh          0
offshore_wind_mwh           0
onshore_wind_mwh            0
solar_mwh                   0
total_wind_mwh              0
renewable_generation_mwh    0
net_load_mwh                0
ExchangeSE_MWh              0
ExchangeGE_MWh              0
ExchangeGreatBelt_MWh       0
PowerToHeatMWh              0
dtype: int64

In [9]:
prod_clean.to_csv(
    "../data/processed/dk2_production_consumption_clean.csv",
    index=False
)